# Ev tapşırığı: Geyim mağazası (Fashion-MNIST + CNN)

Məqsəd: 10 növ geyimi tanıyan CNN qurmaq və testdə **≥ 90% accuracy** almaq.

Notebook-un strukturu: hər addımda əvvəl **izah**, sonra **kod**, sonra **"Düşün" suallarına cavab**.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)  # nəticələr təkrarlana bilsin
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

## 1. Datanı yükləyirik

**İzah.** Dərsdəki `datasets.MNIST` əvəzinə `datasets.FashionMNIST` yazırıq. Forma eynidir (60 000 train, 10 000 test, 28×28, ağ-qara).

Normalizasiya düsturu: `(piksel - MEAN) / STD`. Bunun üçün əvvəlcə train şəkillərinin **özündən** mean və std hesablayırıq.
`train_data.data` 0…255 aralığında `uint8`-dir, ona görə əvvəl 255-ə bölüb 0…1 miqyasına gətiririk (çünki `ToTensor()` də eyni şeyi edir).

In [ ]:
raw_train = datasets.FashionMNIST("data", train=True, download=True)

pixels = raw_train.data.float() / 255      # 0..255 -> 0..1
MEAN = pixels.mean().item()
STD = pixels.std().item()
print(f"MEAN = {MEAN:.4f}, STD = {STD:.4f}   (MNIST-də: 0.1307, 0.3081)")

In [ ]:
base_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])

train_data = datasets.FashionMNIST("data", train=True, download=True, transform=base_tf)
test_data = datasets.FashionMNIST("data", train=False, download=True, transform=base_tf)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000)

print(len(train_data), len(test_data))

### Düşün (Addım 1)

**Bu rəqəmlər haradan gəlir? Yeni dataset üçün saxlamaq olar?**
`0.1307` və `0.3081` MNIST train şəkillərinin bütün piksellərinin (0…1 miqyasında) ortalaması və standart sapmasıdır. Fashion-MNIST-də paltarlar rəqəmlərdən daha çox sahəni doldurur, buna görə dəyərlər fərqlidir (təxminən 0.286 və 0.353). MNIST-in rəqəmlərini olduğu kimi saxlamaq olmaz: normalizasiya girişi "mean≈0, std≈1" vəziyyətinə gətirməlidir, yanlış statistika isə giriş paylanmasını sürüşdürür və təlimi yavaşladır. Ona görə mean/std həmişə **öz train datamızdan** hesablanır (test-dən yox).

## 2. Şəkillərə baxırıq

**İzah.** Label bir rəqəmdir (0…9), insan üçün oxunaqlı deyil. 10 adı bir list-ə yazırıq, label həmin list-in indeksidir, başlıqda `CLASS_NAMES[label]` göstəririk.
Normalizasiyasız (`ToTensor` yalnız) göstəririk ki, şəkillər normal görünsün.

In [ ]:
CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

vis_data = datasets.FashionMNIST("data", train=True, download=True, transform=transforms.ToTensor())

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    img, label = vis_data[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(CLASS_NAMES[label], fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Düşün (Addım 2): təlimdən ƏVVƏL proqnozum

Ən çox qarışacaq class-lar:
1. **Shirt ↔ T-shirt/top** (ikisi də yuxarı geyim, kiçik fərq: yaxa və düymələr 28×28-də itir)
2. **Pullover ↔ Coat** (ikisi də uzun qolludur, siluet demək olar eynidir)
3. **Sneaker ↔ Ankle boot** (və/və ya Sandal ↔ Sneaker: hamısı ayaqqabıdır, forma yaxındır)

Şalvar (Trouser) və çanta (Bag) isə unikal formaya malikdir, asan tanınmalıdır. Bunu 6-cı addımda yoxlayacağam.

## 3. MLP baseline

**İzah.** MLP şəkli düzləyir (28×28 → 784 ədəd) və tam bağlı layer-lərdən keçirir. O, piksellərin **qonşuluğunu bilmir**: 784 piksel ona sadəcə bir siyahıdır.

`train_one_epoch` və `evaluate` funksiyaları model və loader qəbul edir, odur ki, **dəyişmədən** işləyir: data formatı eynidir (1×28×28 tensor + label 0…9), loss də eyni `CrossEntropyLoss`-dur.

> Qeyd: aşağıdakı MLP, optimizer və funksiyalar dərsdəkinin eynisidir. Fərq olarsa, öz dərs kodunuzla əvəz edin.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.flatten(1)          # (B,1,28,28) -> (B,784)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)        # 10 logit (softmax CrossEntropyLoss-un içindədir)


def train_one_epoch(model, loader, optimizer, loss_fn=nn.CrossEntropyLoss()):
    model.train()
    total = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(x)
    return total / len(loader.dataset)      # orta train loss


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
    return 100 * correct / len(loader.dataset)   # accuracy, %


def fit(model, loader, epochs, lr=1e-3):
    """Modeli öyrədir, hər epoch-da (train loss, test acc) çap edir və tarixçəni qaytarır."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for ep in range(1, epochs + 1):
        loss = train_one_epoch(model, loader, opt)
        acc = evaluate(model, test_loader)
        history.append((loss, acc))
        print(f"epoch {ep}/{epochs}  train loss {loss:.4f}  test acc {acc:.2f}%")
    return history

In [ ]:
results = []     # (model, augmentation, epoch, test acc) - son cədvəl üçün
registry = {}    # ad -> (acc, model, factory) - ən yaxşını saxlamaq üçün

torch.manual_seed(42)
mlp = MLP()
hist_mlp = fit(mlp, train_loader, epochs=3)
results.append(("MLP", "yox", 3, hist_mlp[-1][1]))
print(f"\nMLP test accuracy: {hist_mlp[-1][1]:.2f}%")

### Düşün (Addım 3)

**MNIST-də MLP ≈ 97% verirdi. Burada nə gözləyirəm?** **Az**, təxminən 87–89%. Səbəb: rəqəmlər sadə, yüksək kontrastlı xətlərdir. Geyimlərdə isə tekstura, kölgə və bir-birinə çox oxşar class-lar (Shirt/T-shirt/Pullover/Coat) var. Üstəlik MLP piksel qonşuluğunu bilmir.

**`train_one_epoch` və `evaluate` dəyişməlidirmi?** **Xeyr.** Onlar datanın "nə olduğunu" bilmir, yalnız `(x, y)` batch-ləri, model və loss ilə işləyir. Girişin forması (1×28×28) və class sayı (10) eynidir, ona görə eyni funksiyalar işləyir.

## 4. CNN

**İzah.** CNN şəkli düzləmir. `Conv2d` kiçik filtrlərlə (3×3) kənarları, künclər, sonra daha mürəkkəb formaları tapır, `MaxPool` ölçünü yarıya endirir (28→14→7).

Əvvəl bir batch-in shape-ini hər layer-dən sonra çap edirik ki, `fc1`-in giriş ölçüsünə əmin olaq.

In [ ]:
class Net(nn.Module):
    def __init__(self, dropout=0.25):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)    # 1 kanal -> 32
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)   # 32 -> 64
        self.pool = nn.MaxPool2d(2)
        self.drop = nn.Dropout(dropout)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)                              # 10 class

    def forward(self, x, verbose=False):
        if verbose: print("giriş      :", tuple(x.shape))
        x = self.pool(F.relu(self.conv1(x)))
        if verbose: print("conv1+pool :", tuple(x.shape))
        x = self.pool(F.relu(self.conv2(x)))
        if verbose: print("conv2+pool :", tuple(x.shape))
        x = x.flatten(1)
        if verbose: print("flatten    :", tuple(x.shape))
        x = self.drop(F.relu(self.fc1(x)))
        x = self.fc2(x)
        if verbose: print("çıxış      :", tuple(x.shape))
        return x

xb, yb = next(iter(train_loader))
Net().eval()(xb[:8], verbose=True);

In [ ]:
torch.manual_seed(42)
cnn_plain = Net()
hist_cnn = fit(cnn_plain, train_loader, epochs=5)
acc_cnn = hist_cnn[-1][1]
results.append(("CNN", "yox", 5, acc_cnn))
registry["CNN (augmentation-sız)"] = (acc_cnn, cnn_plain, Net)
print(f"\nCNN (augmentation-sız) test accuracy: {acc_cnn:.2f}%")

### Düşün (Addım 4)

**`Net`-də nəyi dəyişmək lazımdır?** **Heç nəyi.** Giriş yenə **1 kanal, 28×28** (Fashion-MNIST də ağ-qaradır), çıxış yenə **10 class**. Ona görə `conv1` (1 kanal), `fc1` (64·7·7) və `fc2` (10) olduğu kimi qalır. Bu, ilk baxışda təəccüblüdür, amma datanın forması MNIST ilə eynidir deyə şəbəkə də eynidir. Dəyişən yalnız **data və label-lərin mənasıdır**.

## 5. Augmentation

**İzah.** Augmentation hər epoch-da train şəkillərinə təsadüfi kiçik dəyişikliklər edir, beləliklə şəbəkə eyni şəkli əzbərləyə bilmir. Seçdiklərim:

- `RandomHorizontalFlip()`: sol↔sağ çevirmə. Geyim demək olar simmetrikdir, ona görə məntiqlidir.
- `RandomAffine(degrees=5, translate=(0.05, 0.05))`: **yumşaq** fırlanma (±5°) və sürüşmə (±5%).
- Vertikal flip **qoymuram** (səbəb aşağıdakı cavabda).

Augmentation yalnız **train**-ə tətbiq olunur. Test həmişə təmiz qalmalıdır ki, ədalətli ölçək. Augmentation-lar `ToTensor`-dan **əvvəl** (PIL şəkil üzərində) yazılır.

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])

train_data_aug = datasets.FashionMNIST("data", train=True, download=True, transform=train_tf)
train_loader_aug = DataLoader(train_data_aug, batch_size=64, shuffle=True)
# test_loader dəyişmir: test_data hələ də base_tf (augmentation-sız) istifadə edir

torch.manual_seed(42)
cnn_aug = Net()
hist_aug = fit(cnn_aug, train_loader_aug, epochs=5)
acc_aug = hist_aug[-1][1]
results.append(("CNN", "bəli", 5, acc_aug))
registry["CNN (augmentation-lı)"] = (acc_aug, cnn_aug, Net)

print(f"\nAugmentation-sız: {acc_cnn:.2f}%   |   Augmentation-lı: {acc_aug:.2f}%")

### Düşün (Addım 5)

**Rəqəmlər üçün horizontal flip niyə pis, geyim üçün olarmı?** Rəqəmdə sol-sağ istiqamət mənanı daşıyır: çevrilmiş `2`, `3`, `5`, `7` artıq həmin rəqəm deyil (və ya heç nə deyil), amma label dəyişmir, yəni şəbəkəyə **yalan** öyrədirik. Geyim isə əksər hallarda sol-sağ simmetrikdir: çevrilmiş futbolka yenə futbolkadır, çevrilmiş ayaqqabı yenə ayaqqabıdır. Bu da realdır, çünki mağaza şəkillərində ayaqqabı hər iki istiqamətə baxa bilər. Deməli geyim üçün olar.

**Vertikal flip?** Yox. Real həyatda başıaşağı şalvar, palto və ya ayaqqabı şəkli görmürük. Test datasında belə şəkillər yoxdur, şəbəkə isə gücünü lazımsız variantları öyrənməyə sərf edər.

**Güclü `RandomAffine` kömək edəcəkmi?** Çox güman **zərər** verər. Fashion-MNIST-də geyimlər onsuz da mərkəzdədir və eyni miqyasdadır, yəni test datası da belədir. Güclü sürüşmə/fırlanma/miqyas train paylanmasını testdən uzaqlaşdırır, və kiçik 28×28 şəkildə obyektin bir hissəsi çərçivədən çıxa bilər. Zəif affine isə zərərsizdir və yüngül regularizasiya verir. Qısa (5 epoch) təlimdə augmentation-lı model bəzən sadəcə **bir qədər yavaş öyrənir**, ona görə nəticə eyni və ya cüzi fərqli ola bilər. Bu normaldır: augmentation əsasən daha uzun təlimdə overfitting-ə qarşı kömək edir.

## 6. Şəbəkə harada səhv edir

**İzah.** Bütün test setini modeldən keçirib proqnozları toplayırıq. Sonra:

1. `show_mistakes`: yalnız səhv tapılan şəkilləri **class adları** ilə çəkir (əsl / proqnoz).
2. **Confusion matrix**: 10×10 cədvəl, sətir = əslində nədir, sütun = şəbəkə nə dedi. Diaqonal = düzgün cavablar, diaqonaldan kənar hər xana = konkret qarışdırma.

In [ ]:
@torch.no_grad()
def predict_all(model, loader):
    model.eval().to(device)
    preds, labels = [], []
    for x, y in loader:
        preds.append(model(x.to(device)).argmax(1).cpu())
        labels.append(y)
    return torch.cat(preds), torch.cat(labels)

# Analiz üçün 4-5-dən ən yaxşı CNN-i götürürük
best6 = max(["CNN (augmentation-sız)", "CNN (augmentation-lı)"], key=lambda k: registry[k][0])
print("Analiz olunan model:", best6)
preds, labels = predict_all(registry[best6][1], test_loader)

test_raw = datasets.FashionMNIST("data", train=False, download=True, transform=transforms.ToTensor())


def show_mistakes(preds, labels, n=16):
    wrong = (preds != labels).nonzero().squeeze(1)[:n]
    fig, axes = plt.subplots(4, 4, figsize=(9, 9))
    for ax, idx in zip(axes.flat, wrong):
        img, _ = test_raw[idx.item()]
        ax.imshow(img.squeeze(), cmap="gray")
        ax.set_title(f"Əsl: {CLASS_NAMES[labels[idx]]}\nProqnoz: {CLASS_NAMES[preds[idx]]}", fontsize=9, color="crimson")
        ax.axis("off")
    for ax in axes.flat[len(wrong):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_mistakes(preds, labels)

In [ ]:
confusion = torch.zeros(10, 10, dtype=torch.int64)
for l, p in zip(labels, preds):
    confusion[l, p] += 1            # sətir = əsl, sütun = proqnoz

plt.figure(figsize=(9, 8))
plt.imshow(confusion, cmap="Blues")
plt.xticks(range(10), CLASS_NAMES, rotation=45, ha="right")
plt.yticks(range(10), CLASS_NAMES)
plt.xlabel("Şəbəkə nə dedi (proqnoz)")
plt.ylabel("Əslində nədir")
plt.title("Confusion matrix")
for i in range(10):
    for j in range(10):
        v = confusion[i, j].item()
        plt.text(j, i, v, ha="center", va="center", fontsize=8,
                 color="white" if v > confusion.max() / 2 else "black")
plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
# Ən çox qarışan cütlər (hər iki istiqamət cəmlənir: A->B və B->A)
pairs = []
for i in range(10):
    for j in range(i + 1, 10):
        pairs.append((confusion[i, j].item() + confusion[j, i].item(), CLASS_NAMES[i], CLASS_NAMES[j]))
pairs.sort(reverse=True)

print("Ən çox qarışdırılan 3 cüt:")
for count, a, b in pairs[:3]:
    print(f"  {a} <-> {b}: {count} səhv")

print("\nHər class üzrə accuracy:")
per_class = confusion.diag().float() / confusion.sum(1).float() * 100
for name, acc in sorted(zip(CLASS_NAMES, per_class.tolist()), key=lambda t: -t[1]):
    print(f"  {name:12s} {acc:5.1f}%")

### Düşün (Addım 6)

> Yuxarıdakı çıxışa baxıb bu hissəni öz nəticənizlə yoxlayın. Aşağıdakılar tipik nəticədir.

**Ən çox qarışdırılan 3 cüt:** adətən **Shirt ↔ T-shirt/top**, **Shirt ↔ Pullover** (və/və ya **Pullover ↔ Coat**). Hamısı yuxarı geyimdir. Proqnozum (addım 2) əsasən düz çıxır: yuxarı geyim qrupu doğrudan da ən çox qarışır. Ayaqqabı qrupu (Sneaker ↔ Ankle boot / Sandal) isə proqnozdakından **daha az** qarışır, şəbəkə bu cütləri yaxşı ayırır.

**Səhv şəkilləri özüm tanıyıram?** Çoxunu **çətin** tanıyıram. 28×28 qara-ağ şəkildə yaxa, düymə, fermuar kimi detallar itib, Shirt ilə T-shirt-in fərqi çox vaxt göz üçün də aydın olmur. Deməli bəzi səhvlər şəbəkənin yox, datanın aşağı keyfiyyətinin nəticəsidir. Bir neçə şəkildə isə label-in özü mübahisəlidir.

**Ən asan tanınan class:** **Trouser** (adətən ≈ 98–99%), sonra **Bag** və **Sneaker/Ankle boot**. Səbəb: şalvarın siluet **unikal**dır (iki uzun ayrı paça), digər class-larla qarışmaq çətindir. Çantanın forması da (qulplu düzbucaqlı) başqa class-lara bənzəmir. Ən çətin class isə **Shirt** (≈ 70–75%): qalan yuxarı geyimlərin hamısına bənzəyir.

## Bonus A. Modeli təkmilləşdirmək

**İzah.** Qayda: **hər dəfə yalnız bir şey dəyişirik**, çünki iki şeyi birdən dəyişsək, hansının işlədiyini bilmərik.

- **Dəyişiklik 1:** üçüncü conv layer (128 filtr) əlavə edirik. Digər hər şey eynidir (5 epoch, eyni augmentation).
- **Dəyişiklik 2:** yenə eyni şəbəkə, amma **epoch 5 → 12**.

Üçüncü `MaxPool`-dan sonra ölçü: 28 → 14 → 7 → **3** (7/2 = 3,5 və PyTorch aşağı yuvarlaqlaşdırır). Deməli `fc1` giriş ölçüsü `128·3·3 = 1152` olur.

In [ ]:
class Net3(nn.Module):
    def __init__(self, dropout=0.25):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.drop = nn.Dropout(dropout)
        self.fc1 = nn.Linear(128 * 3 * 3, 128)    # 28->14->7->3
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 14x14
        x = self.pool(F.relu(self.conv2(x)))   # 7x7
        x = self.pool(F.relu(self.conv3(x)))   # 3x3
        x = x.flatten(1)
        x = self.drop(F.relu(self.fc1(x)))
        return self.fc2(x)

# Dəyişiklik 1: yalnız conv3 (5 epoch, augmentation ilə)
torch.manual_seed(42)
net3_5 = Net3()
hist3_5 = fit(net3_5, train_loader_aug, epochs=5)
results.append(("CNN + conv3", "bəli", 5, hist3_5[-1][1]))
registry["CNN + conv3 (5 epoch)"] = (hist3_5[-1][1], net3_5, Net3)

In [ ]:
# Dəyişiklik 2: eyni Net3, epoch 12
torch.manual_seed(42)
net3_12 = Net3()
hist3_12 = fit(net3_12, train_loader_aug, epochs=12)
results.append(("CNN + conv3", "bəli", 12, hist3_12[-1][1]))
registry["CNN + conv3 (12 epoch)"] = (hist3_12[-1][1], net3_12, Net3)

# train loss və test accuracy bir qrafikdə
epochs = range(1, len(hist3_12) + 1)
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(epochs, [h[0] for h in hist3_12], "o-", color="tab:red", label="train loss")
ax1.set_xlabel("epoch"); ax1.set_ylabel("train loss", color="tab:red")
ax2 = ax1.twinx()
ax2.plot(epochs, [h[1] for h in hist3_12], "s-", color="tab:blue", label="test acc")
ax2.set_ylabel("test accuracy (%)", color="tab:blue")
plt.title("Train loss düşür, test accuracy nə edir?")
plt.tight_layout()
plt.show()

### Düşün (Bonus A)

**Üçüncü MaxPool-dan sonra ölçü və `fc1`:** 28 → 14 → 7 → **3×3** (7-nin yarısı 3,5-dir, PyTorch aşağı yuvarlaqlaşdırır, son sətir/sütun atılır). Son conv-un çıxışı 128 kanaldır, ona görə `fc1`-in giriş ölçüsü **128 · 3 · 3 = 1152** olmalıdır (əvvəl `64 · 7 · 7 = 3136` idi).

**Train loss düşür, test accuracy artıq artmır, bu nə deməkdir?** Bu **overfitting**-dir (Lesson4-də danışdığımız): şəbəkə train şəkillərini "əzbərləməyə" başlayır, yəni train-də getdikcə yaxşılaşır, amma yeni (test) şəkillərə ümumiləşdirə bilmir. Həll yolları: Dropout-u artırmaq, augmentation, daha az epoch (early stopping) və ya daha kiçik/sadə model.

## 7. Modeli saxlayırıq

**İzah.** Bütün modeli yox, yalnız **`state_dict`** saxlayırıq: bu, layer-lərin adı → çəki tensoru lüğətidir (conv/fc çəkiləri və biasları).

Yükləmək üçün isə əvvəlcə **eyni arxitekturalı boş modeli** yaradırıq (`Net()` və ya `Net3()`), sonra `load_state_dict` ilə çəkiləri içinə yazırıq. Yüklədikdən sonra `.eval()` çağırmaq vacibdir (Dropout söndürülsün), yoxsa accuracy eyni çıxmaz.

In [ ]:
best_name = max(registry, key=lambda k: registry[k][0])
best_acc, best_model, best_factory = registry[best_name]
print(f"Ən yaxşı model: {best_name}  ->  {best_acc:.2f}%")

torch.save(best_model.state_dict(), "fashion_cnn.pt")

# Yeni boş model yaradıb çəkiləri yükləyirik
loaded = best_factory().to(device)
loaded.load_state_dict(torch.load("fashion_cnn.pt", map_location=device))
loaded.eval()

loaded_acc = evaluate(loaded, test_loader)
print(f"Saxlanmazdan əvvəl: {best_acc:.2f}%   |   Yüklədikdən sonra: {loaded_acc:.2f}%")
assert abs(best_acc - loaded_acc) < 1e-6, "Accuracy eyni olmalıdır!"
results.append(("Ən yaxşı model (" + best_name + ")", "-", "-", best_acc))

### Düşün (Addım 7)

**Niyə yalnız `state_dict`?** Çünki o yalnız **çəkiləri** (ədədləri) saxlayır, kod yox. `torch.save(model)` isə modeli `pickle` ilə, class-ın yoluna bağlı saxlayır: class adı/faylı dəyişəndə yükləmək sınır, fayl böyükdür və başqa mənbədən gələn pickle faylı təhlükəli ola bilər. `state_dict` daha kiçik, daha portativ və təhlükəsizdir.

**Yükləyəndə nə lazım olur?** Eyni arxitekturanın **class kodu** (`Net`/`Net3`): əvvəl boş model yaradılır (layer adları və ölçüləri saxlanmış lüğətlə üst-üstə düşməlidir), sonra `load_state_dict(...)` və `model.eval()`.

## Özünü yoxla: nəticələr cədvəli

In [ ]:
print(f"{'Model':45s} {'Augmentation':13s} {'Epoch':6s} {'Test accuracy (%)':s}")
print("-" * 85)
for model, aug, ep, acc in results:
    print(f"{model:45s} {aug:13s} {str(ep):6s} {acc:.2f}")

print("\nCNN ≥ 90% ?", "BƏLİ" if max(a for m, _, _, a in results if m.startswith("CNN")) >= 90 else "XEYR")

**Nəticələrin şərhi.** Cədvəl yuxarıdakı kodun çıxışından doldurulur. MLP baseline-dan CNN-in yaxşı olduğu görünməlidir. Əgər augmentation 5 epoch-da cüzi fərq verirsə (və ya bir az aşağıdırsa), bu o deməkdir ki, qısa təlimdə şəbəkə hələ overfit etmir və augmentation-ın faydası uzun təlimdə (Bonus A, 12 epoch) görünür.

**Təhvil verməzdən əvvəl yoxlama siyahısı:**
- [ ] Notebook-u `Kernel → Restart & Run All` ilə əvvəldən axıra error-suz işə salmısınız
- [ ] Mean/std özünüz hesablanıb (MNIST-dən köçürülməyib)
- [ ] Qrafiklərin başlıqlarında class adları var
- [ ] Confusion matrix çəkilib, ən çox qarışan cütlər yazılıb (6-cı addımın cavabını öz çıxışınızla yoxlayın)
- [ ] CNN test-də ≥ 90%
- [ ] `fashion_cnn.pt` saxlanılıb və yenidən yüklənir